In [ ]:
import os
import json
import random
import h5py
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
def read_valid_indices(data_path):
    def load_indices(desc_path, mode):
        with open(desc_path, "r") as f:
            desc = json.load(f)
        if mode is None:
            return desc["valid_indices"]
        elif '_past_feat_dynamic_real' == mode:
            return desc['past_feat_dynamic_real_valid_indices']
        elif '_feat_dynamic_real' == mode:
            return desc['feat_dynamic_real_valid_indices']
        else:
            raise NotImplementedError

    dataset_part_name = data_path.split('/')[-1].split(".npy")[0]

    if os.path.exists(data_path.replace(".npy", ".json")):
        return load_indices(data_path.replace(".npy", ".json"), None)

    elif '_past_feat_dynamic_real' in dataset_part_name:
        desc_name = (
            dataset_part_name.split('_past_feat_dynamic_real')[0]
            + dataset_part_name.split('_past_feat_dynamic_real')[1]
            + ".json"
        )
        desc_path = os.path.join(os.path.dirname(data_path), desc_name)
        return load_indices(desc_path, '_past_feat_dynamic_real')

    elif '_feat_dynamic_real' in dataset_part_name:
        desc_name = (
            dataset_part_name.split('_feat_dynamic_real')[0]
            + dataset_part_name.split('_past_feat_dynamic_real')[1]
            + ".json"
        )
        desc_path = os.path.join(os.path.dirname(data_path), desc_name)
        return load_indices(desc_path, '_feat_dynamic_real')

    elif '_values' in dataset_part_name:
        desc_name = dataset_part_name.split('_values')[0] + ".json"
        desc_path = os.path.join(os.path.dirname(data_path), desc_name)
        return load_indices(desc_path, None)

    elif '_data' in dataset_part_name:
        desc_name = dataset_part_name.split('_data')[0] + ".json"
        desc_path = os.path.join(os.path.dirname(data_path), desc_name)
        return load_indices(desc_path, None)

    else:
        raise AssertionError(f"dataset_part_name: {dataset_part_name}")


In [ ]:
def visualize_cell(
    cell_id: int,
    h5_path: str,
    codebook_index_json: str,
    K: int = 8,
    max_len: int = 4096,
    seed: int = 0,
):
    """
    从指定 cell 中随机取 K 条，回溯原始时间序列并可视化
    """

    random.seed(seed)
    np.random.seed(seed)

    # ---------- load file_id -> codebook path ----------
    with open(codebook_index_json, "r", encoding="utf-8") as f:
        raw = json.load(f)
    file_index = {int(k): v for k, v in raw.items()}

    # ---------- load cell usage ----------
    with h5py.File(h5_path, "r") as f:
        dset = f[str(cell_id)]
        total = dset.shape[0]
        if total == 0:
            raise ValueError(f"Cell {cell_id} is empty")

        sel_idx = np.random.choice(total, size=min(K, total), replace=False)

        order = np.argsort(sel_idx)
        sel_idx_sorted = sel_idx[order]

        rows_sorted = dset[sel_idx_sorted]
    rows = rows_sorted[np.argsort(order)]   # (K,4)


    # ---------- plot ----------
    fig, axes = plt.subplots(len(rows), 1, figsize=(12, 2.5 * len(rows)), sharex=True)
    if len(rows) == 1:
        axes = [axes]

    for i, (file_id, n, c, s, l) in enumerate(rows):
        codebook_path = file_index[int(file_id)]

        # 替换路径
        data_path = (
            codebook_path
            .replace("<CODEBOOK_OUTPUT_ROOT>/",
                     "<RAW_DATA_ROOT>/")
            .replace("_codebook.npz", ".npy")
        )

        if not os.path.exists(data_path):
            raise FileNotFoundError(data_path)

        # ---------- load raw data ----------
        raw_data = np.load(data_path)     # shape [N, C, L] or [N, C]
        valid_indices = read_valid_indices(data_path)

        start, end = valid_indices[n][c]
        ts = raw_data[n, c, start:end][:max_len]

        axes[i].plot(ts, linewidth=1.2)
        axes[i].set_title(
            f"cell={cell_id} | file_id={file_id} | n={n} c={c} level={s}",
            fontsize=10
        )
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("time index")
    plt.tight_layout()
    plt.show()


In [ ]:
CELL_ID = 2300
H5_PATH = "cell_usage_uint32_2500.h5"
CODEBOOK_INDEX_JSON = "registered_codebook_index.json"

visualize_cell(
        cell_id=CELL_ID,
        h5_path=H5_PATH,
        codebook_index_json=CODEBOOK_INDEX_JSON,
        K=6,
        max_len=4096,
        seed=1,
    )